<h1 align=center style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
تشخیص اخبار دروغین
</font>
</h1>

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
مقدمه و صورت مسئله
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
مطمئنا بارها شنیده‌اید که در فضای مجازی اخبار دروغ و شایعه بسیار رایج است. افرادی هستند که به هر دلیل اقدام به انتشار خبرهای دروغ می‌کنند. برخی این دروغ‌ها از جنس خبرهای سیاسی، اقتصادی یا حواشی راجع به زندگی افراد مشهور و سلبریتی‌هاست. اما به هر نحوی که باشد، دروغ رفتار خوبی نیست.
    <br>
    پس به عنوان یک مخاطب خوب است که بتوانیم تشخیص دهیم چه خبری حقیقت و چه خبری دروغ است. اگر از دید یک خبرگزاری هم به ماجرا نگاه کنیم، متوجه خواهیم شد که اصلا دوست نداریم یک خبر دروغ اقتصادی یا سیاسی از زبان ما منتشر شود؛ در غیر این صورت عواقب پیشبینی‌نشده‌ای در انتظارمان خواهد بود!
    <br>
    پس حال که به اهمیت و کاربرد تشخیص اخبار دروغین پی برده‌ایم، خوب است خودمان تلاش کنیم مدلی آموزش دهیم که خبرهای دروغ را از حقیقت جدا کنیم.
    <br>
    البته تشخیص این امر، وابسته به توانایی پردازش زبان طبیعی است و با دانش یادگیری ماشین کلاسیک، به سادگی نمی‌توان مسئله رو حل کرد. برای اینکه مسئله قابل حل باشد، بخش «پردازش زبان طبیعی» را خود ما انجام داده‌ایم و نتیجه آن‌ را در قالب بخشی از مجموعه‌داده در اختیار شما قرار داده‌ایم.
</font>
</p>

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
وارد کردن کتابخانه‌های مورد نیاز
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    ابتدا کتابخانه‌های مورد نیازتان را وارد کنید.
</font>
</p>

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp

from urllib.parse import urlparse

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.svm import SVC
from sklearn.metrics import (
    f1_score,
    classification_report,
    confusion_matrix
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
معرفی مجموعه داده
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
مجموعه‌داده این تمرین، مانند تمرین‌های قبلی به دو مجموعه آموزش و آزمون تقسیم‌بندی می‌شود. اما نکته حائز اهمیت این است که هر کدام از این دو مجموعه آزمون و آموزش، خود دارای دو مجموعه داده است. به عبارت بهتر مجموعه آموزش دارای دو مجموعه‌داده به اسم <code>news_train.csv</code> و <code>news_train_text_vectors.npz</code> است (مجموعه آزمون هم همینطور است).
    <br>
    در فایل <code>news_train.csv</code> اطلاعات کلی مانند زمان انتشار، منبع و برچسب خبر آمده است؛ اما خود خبر را در این مجموعه‌داده نداریم. بلکه خود خبر را می‌توان در فایل <code>news_train_text_vectors.npz</code> جستجو کرد. اما در <code>news_train_text_vectors.npz</code> متن خبر قرار نگرفته است، بلکه به کمک روش‌های پردازش زبان طبیعی، متن هر خبر را به یک بردار ۴۲۱۴۱ المانی تبدیل کرده‌ایم و به ازای هر خبر، یک بردار با ابعاد <code>(42141, 1)</code> وجود دارد.
    <br>
    توضیحات مجموعه‌داده <code>news_train.csv</code> را در جدول زیر می‌توانید مشاهده کنید.
</font>
</p>

<center>
<div dir=rtl style="direction: rtl;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazirmatn" size=3>
    
|ستون|توضیحات|
|:------:|:---:|
|author|شخصی که متن خبر را نوشته است|
|published|تاریخ و ساعتی که خبر منتشر شده است|
|site_url|سایت منتشرکننده خبر|
|type|برای هر خبر یک دسته یا یک نوعی نیز ذخیره کرده‌ایم. این ستون می‌تواند یکی از مقادیر `bs` به معنی *پرت* یا ‍‍‍‍`conspiracy` به معنی *توطئه آمیز* یا `bias` به معنی *سوگیری یا متعصبانه* یا `hate` به معنی *نفرت‌افکن یا خشم‌آلود* یا ‍`satrie` به معنی *طنز* یا `junksci` به معنی *به درد نخور یا آشغال* و `fake` به معنی *دروغ* باشد|
|label|برچسب خبر را نشان می‌دهد. این ستون یا `Fake` است یا `Real`|
    
</font>
</div>
</center>


<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    در کنار فایل <code>news_train.csv</code> فایل <code>news_train_text_vectors.npz</code> نیز قرار دارد. کلیات این فایل را در قسمت قبلی توضیح دادیم. به حجیم بودن فایل و صرفه‌جویی درفضای ذخیره‌سازی و کاهش زمان برای بارگذاری فایل، به جای فرمت <code>csv</code> آن را در فرمت <code>npz</code> در اختیار شما قرار دادیم. اما پیشنهاد می‌کنیم برای راحتی خودتان، پس از خواندن فایل، آن را به یک دیتافریم با ۱۵۰۰ سطر و ۴۲۱۴۱ ستون تبدیل کنید.
    <br>
    اینطور در نظر داشته باشید که متن هر خبر را با ۴۲۱۴۱ ویژگی برایتان توصیف کرده‌ایم!
</font>
</p>


<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
خواندن مجموعه داده
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    در ابتدا نیاز است فایل‌های مجموعه‌داده را بخوانید. نمونه‌های آموزشی در فایل <code>news_train.csv</code> و <code>news_train_text_vectors.npz</code> و نمونه‌های آزمون که باید دسته‌ی آن‌ها را پیش‌بینی کنید در فایل <code>news_test.csv</code> و <code>news_test_text_vectors.npz</code> ذخیره شده‌اند. اگر لازم دانستید می‌توانید به دلخواه خود بخشی از دادگان آموزشی را به عنوان دادگان اعتبارسنجی نیز جدا کنید.
</font>
</p>

In [ ]:
data_path = '/content/drive/MyDrive/ML/Quera_exercise/تشخیص اخباردروغین/'

train_data = pd.read_csv(data_path + 'news_train.csv')
test_data = pd.read_csv(data_path + 'news_test.csv')

train_text_vectors = sp.load_npz(
    data_path + 'news_train_text_vectors.npz'
)

test_text_vectors = sp.load_npz(
    data_path + 'news_test_text_vectors.npz'
)

In [ ]:
train_data

,author,published,site_url,type,label
0,No Author,2016-11-01T03:28:50.389+02:00,clickhole.com,satire,Fake
1,Anonymous,2016-10-27T21:30:00.000+03:00,abeldanger.net,bs,Fake
2,Kali74,2016-10-27T02:54:49.093+03:00,abovetopsecret.com,bs,Fake
3,Alex Ansary,2016-11-04T22:44:06.026+02:00,amtvmedia.com,bs,Fake
4,Luke Stranahan,2016-11-23T15:10:56.702+02:00,returnofkings.com,hate,Real
...,...,...,...,...,...
1495,Daniel Greenfield,2016-11-02T13:25:57.336+02:00,frontpagemag.com,hate,Real
1496,theeconomiccollapseblog.com,http://theeconomiccollapseblog.com/wp-content/...,theeconomiccollapseblog.com,bs,Fake
1497,Alex Ansary,2016-11-02T19:24:00.901+02:00,amtvmedia.com,bs,Fake
1498,EdJenner,2016-11-22T15:44:03.678+02:00,dailywire.com,bias,Real


In [ ]:
test_data

,author,published,site_url,type
0,Barracuda Brigade,2016-11-23T21:54:00.000+02:00,100percentfedup.com,bias
1,No Author,2016-11-15T18:16:59.191+02:00,clickhole.com,satire
2,Lisa Beach,2016-10-26T22:46:37.612+03:00,westernjournalism.com,bias
3,Rich Knoch,2016-10-26T22:46:18.548+03:00,westernjournalism.com,bias
4,Bruce Thornton,2016-11-01T07:32:54.546+02:00,frontpagemag.com,hate
...,...,...,...,...
341,No Author,2016-11-16T22:46:52.808+02:00,clickhole.com,satire
342,Sarah Jones,2016-11-01T17:55:04.779+02:00,politicususa.com,bias
343,Anonymous,2016-10-26T22:19:00.000+03:00,abeldanger.net,bs
344,The Alex Jones Show,2016-10-28T22:17:00.659+03:00,infowars.com,conspiracy


In [ ]:
train_text_vectors

<COOrdinate sparse matrix of dtype 'float64'
	with 284524 stored elements and shape (1500, 42141)>

In [ ]:
test_text_vectors

<COOrdinate sparse matrix of dtype 'float64'
	with 52811 stored elements and shape (346, 42141)>

In [ ]:
print('Train data shape:', train_data.shape)
print('Test data shape:', test_data.shape)
print('---------------------------------------------------------------------------')
print('Train text vectors shape:', train_text_vectors.shape)
print('Test text vectors shape:', test_text_vectors.shape)
print('---------------------------------------------------------------------------')
print('\nMissing values in train data:')
print(train_data.isnull().sum())
print('---------------------------------------------------------------------------')
print('\nTarget distribution:')
print(train_data['label'].value_counts(normalize=True))
print('---------------------------------------------------------------------------')
print('Number of training CSV rows:', len(train_data))
print('Number of training text vectors:', train_text_vectors.shape[0])
print('---------------------------------------------------------------------------')
print('\nNumber of test CSV rows:', len(test_data))
print('Number of test text vectors:', test_text_vectors.shape[0])
print('---------------------------------------------------------------------------')
print('\nNumber of training text features:', train_text_vectors.shape[1])
print('Number of test text features:', test_text_vectors.shape[1])

Train data shape: (1500, 5)
Test data shape: (346, 4)
---------------------------------------------------------------------------
Train text vectors shape: (1500, 42141)
Test text vectors shape: (346, 42141)
---------------------------------------------------------------------------

Missing values in train data:
author       0
published    0
site_url     0
type         0
label        0
dtype: int64
---------------------------------------------------------------------------

Target distribution:
label
Fake    0.658667
Real    0.341333
Name: proportion, dtype: float64
---------------------------------------------------------------------------
Number of training CSV rows: 1500
Number of training text vectors: 1500
---------------------------------------------------------------------------

Number of test CSV rows: 346
Number of test text vectors: 346
---------------------------------------------------------------------------

Number of training text features: 42141
Number of test text fe

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
پیش‌پردازش و مهندسی ویژگی
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    در این سوال شما می‌توانید از هر تکنیک پیش‌پردازش/مهندسی ویژگی که در فصل‌های گذشته آموختید، استفاده کنید.
    <br>
    تکنیک‌هایی که استفاده می‌کنید به شکل مستقیم مورد ارزیابی توسط سامانه داوری قرار <b>نمی‌گیرند.</b> بلکه همه آن‌ها در دقت مدل شما تاثیر خواهند گذاشت؛ بنابراین هر چه پیش‌پردازش/مهندسی ویژگی بهتری انجام دهید تا دقت مدل بهبود پیدا کند، امتیاز بیشتری از این سوال کسب خواهید کرد.
    <br>
    از آنجایی که قرار است در این تمرین از الگوریتم ماشین بردار پشتیبان استفاده کنید، حتما باید مجموعه‌داده <code>news_train_text_vectors.npz</code> را با روش‌های کاهش ابعادی که یاد گرفته‌اید، به نحوی تغییر دهید که در زمان معقولی توسط الگوریتم <code>svm</code> پردازش شود. اگر چه الگوریتمی که برای کاهش ابعاد استفاده می‌کنید در کوئرا بررسی نمی‌شود، اما پیشنهاد ما استفاده از <code>PCA</code> است. آنچه اهمیت دارد آن است که بین کوچک‌کردن داده و کیفیت مدل تعادل برقرار کنید. طبیعی است که هرچقدر داده کوچک‌تر باشد، احتمالا الگوریتم ماشین بردار پشتیبان زودتر به نتیجه می‌رسد؛ اما در نظر داشته باشید با کاهش بعد، شما عملا بخشی از داده را از دست می‌دهید!
</font>
</p>

<h3 align=right style="direction: rtl;text-align: right;line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
    استفاده از <code>scikit-learn</code>
</font>
</h3>


<p dir=rtl style="direction: rtl;text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazirmatn" size=3>
    الگوریتم <i>تحلیل مولفه‌های اصلی</i> با نام <code>PCA</code> در پکیج <code>decomposition</code> این کتابخانه در دسترس است. برخی از آرگومان‌های مهم آن در جدول زیر آمده است، اما جهت مطالعه‌ی کامل‌تر مستندات می‌توانید به <a href="https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html" target="_blank">این لینک</a> مراجعه فرمایید.
    <br>
    برای استفاده از این الگوریتم، می‌توانید با اجرای <code>from sklearn.decomposition import PCA</code> آن را وارد (<code>import</code>) کنید.
    <br>
    
</font>
</p>


<center>
<div dir=rtl style="direction: rtl;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazirmatn" size=3>

|آرگومان|جنس و تایپ|توضیحات|
|:------:|:--------:|:---:|
|n_components|<code>int</code> یا <code>float</code>|تعداد مولفه‌هایی است که قصد داریم پس از اجرای الگوریتم، آن‌ها را داشته باشیم و حذف نشوند. اگر به این آرگومان مقدار ندهیم، همه مولفه‌ها حفظ خواهند شد. اگر جنس این آرگومان <code>int</code> باشد، نشانگر تعداد مولفه‌هایی است که قصد داریم حفظ شود. اما اگر از جنس <code>float</code> و به شکل $0< n\_components< 1$ باشد، به این معنا خواهد بود که به اندازه‌ای از مولفه‌ها حفظ شوند که واریانس داده‌ها بزرگتر از `n_componets`٪ باشد. بنابراین می‌توان اینگونه برداشت کرد که هرچه این عدد به ۱ نزدیک‌تر باشد، چون قرار است داده واریانس زیادی داشته باشد، پس مولفه‌های بیشتری حفظ می‌شوند.|
    
</font>
</div>
</center>



<p dir=rtl style="direction: rtl;text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazirmatn" size=3>
    کلاس <code>PCA</code> دارای سه متد به شرح زیر است:
</font>
</p>


<center>
<div dir=rtl style="direction: rtl;text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazirmatn" size=3>
    
|اسم متد|توضیحات|
|:------:|:---:|
|fit(X)|الگوریتم را با X که در ابعاد `(n_samples, n_features)` است|
|transform(X)|این تابع X که از ابعاد `(n_samples, n_features)` است را به یک ماتریس دیگری با ابعاد `(n_samples, n_components)` تبدیل می‌کند. پس خروجی این تابع یک ماتریس به ابعاد `(n_samples, n_components)` خواهد بود|
|fit_transform(X)|ابتدا تابع <code>fit</code> و سپس تابع <code>transform</code> را اجرا می‌کند|
</font>
</div>
</center>

Feature Engineering: Temporal Features

In [ ]:
# Convert publication dates to datetime
for df in [train_data, test_data]:
    df['published'] = pd.to_datetime(
        df['published'],
        errors='coerce',
        format='mixed',
        utc=True
    )

    # Extract temporal features
    df['year'] = df['published'].dt.year
    df['month'] = df['published'].dt.month
    df['hour'] = df['published'].dt.hour
    df['day_of_week'] = df['published'].dt.dayofweek
    df['weekend'] = (df['day_of_week'] >= 5).astype(int)

# Remove the original publication date
train_data.drop(columns=['published'], inplace=True)
test_data.drop(columns=['published'], inplace=True)

print("Training data shape:", train_data.shape)
print("Test data shape:", test_data.shape)

print("\nMissing temporal values in training data:")
print(train_data[['year', 'month', 'hour', 'day_of_week']].isnull().sum())

print("\nMissing temporal values in test data:")
print(test_data[['year', 'month', 'hour', 'day_of_week']].isnull().sum())

Training data shape: (1500, 9)
Test data shape: (346, 8)

Missing temporal values in training data:
year           1
month          1
hour           1
day_of_week    1
dtype: int64

Missing temporal values in test data:
year           0
month          0
hour           0
day_of_week    0
dtype: int64


Feature Preparation

In [ ]:
# Extract website domains
for df in [train_data, test_data]:
    df['site_domain'] = df['site_url'].apply(
        lambda x: urlparse(str(x)).netloc.lower()
    )

# Remove the original website URL
train_data.drop(columns=['site_url'], inplace=True)
test_data.drop(columns=['site_url'], inplace=True)

# Separate features and target
X = train_data.drop(columns=['label'])
y = train_data['label'].map({'Fake': 0, 'Real': 1})

X_test = test_data.copy()

# Define feature types
categorical_features = ['author', 'type', 'site_domain']
numeric_features = ['year', 'month', 'hour', 'day_of_week', 'weekend']

print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)
print("\nTarget values:")
print(y.value_counts())
print("\nTraining features shape:", X.shape)
print("Test features shape:", X_test.shape)

Categorical features: ['author', 'type', 'site_domain']
Numeric features: ['year', 'month', 'hour', 'day_of_week', 'weekend']

Target values:
label
0    988
1    512
Name: count, dtype: int64

Training features shape: (1500, 8)
Test features shape: (346, 8)


In [ ]:
# Convert text matrices to CSR format for row indexing
train_text_vectors = train_text_vectors.tocsr()
test_text_vectors = test_text_vectors.tocsr()

# Split row indices to keep tabular and text data aligned
indices = np.arange(len(X))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.25,
    random_state=42,
    stratify=y
)

X_train = X.iloc[train_idx].copy()
X_val = X.iloc[val_idx].copy()

y_train = y.iloc[train_idx]
y_val = y.iloc[val_idx]

text_train = train_text_vectors[train_idx]
text_val = train_text_vectors[val_idx]

print("Training samples:", len(train_idx))
print("Validation samples:", len(val_idx))
print("Training text shape:", text_train.shape)
print("Validation text shape:", text_val.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nValidation target distribution:")
print(y_val.value_counts())

Training samples: 1125
Validation samples: 375
Training text shape: (1125, 42141)
Validation text shape: (375, 42141)

Training target distribution:
label
0    741
1    384
Name: count, dtype: int64

Validation target distribution:
label
0    247
1    128
Name: count, dtype: int64


In [ ]:
# Define preprocessing pipelines
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_features),
        ('num', numeric_transformer, numeric_features)
    ],
    sparse_threshold=1.0
)

# Fit only on training data
X_train_tabular = preprocessor.fit_transform(X_train)

# Transform validation data without refitting
X_val_tabular = preprocessor.transform(X_val)

print("Training tabular features shape:", X_train_tabular.shape)
print("Validation tabular features shape:", X_val_tabular.shape)

Training tabular features shape: (1125, 356)
Validation tabular features shape: (375, 356)


In [ ]:
# Reduce text dimensionality using TruncatedSVD
svd = TruncatedSVD(
    n_components=300,
    random_state=42
)

text_train_reduced = svd.fit_transform(text_train)
text_val_reduced = svd.transform(text_val)

# Calculate cumulative explained variance
cumulative_variance = np.cumsum(svd.explained_variance_ratio_)

print("Reduced training text shape:", text_train_reduced.shape)
print("Reduced validation text shape:", text_val_reduced.shape)
print(
    "Cumulative explained variance with 300 components:",
    round(cumulative_variance[-1], 4)
)

# Check whether 95% variance was retained
if cumulative_variance[-1] >= 0.95:
    n_components_95 = np.searchsorted(cumulative_variance, 0.95) + 1
    print("Components needed for 95% variance:", n_components_95)
else:
    print("95% variance was not reached with 300 components.")

Reduced training text shape: (1125, 300)
Reduced validation text shape: (375, 300)
Cumulative explained variance with 300 components: 0.4791
95% variance was not reached with 300 components.


In [ ]:
# Combine tabular and reduced text features
X_train_combined = sp.hstack([
    sp.csr_matrix(X_train_tabular),
    sp.csr_matrix(text_train_reduced)
]).tocsr()

X_val_combined = sp.hstack([
    sp.csr_matrix(X_val_tabular),
    sp.csr_matrix(text_val_reduced)
]).tocsr()

print("Combined training shape:", X_train_combined.shape)
print("Combined validation shape:", X_val_combined.shape)

Combined training shape: (1125, 656)
Combined validation shape: (375, 656)


<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
مدل‌سازی
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    حال که داده را پاکسازی کرده و احتمالا ویژگی‌هایی را به آن افزوده یا از آن حذف کرده‌اید، وقت آن است که مدلی آموزش دهید که بتواند متغیر هدف این مسئله را پیش‌بینی کند.
    <br>
    <b>توجه داشته باشید که برای پیشبینی متغیر هدف این مسئله حتما باید از الگوریتم <code>SVM</code> استفاده کنید. در غیر این صورت، نمره‌ای دریافت نخواهید کرد.</b>
    <br>
    نحوه استفاده از این الگوریتم را در تمرین‌ها و درسنامه‌های قبلی آموخته‌اید.
</font>
</p>

<h3 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
آموزش مدل
</font>
</h3>

In [ ]:
# Train SVM classifier
svm_model = SVC(
    kernel='linear',
    C=1.0,
    class_weight='balanced'
)

svm_model.fit(X_train_combined, y_train)

# Predict validation labels
y_pred = svm_model.predict(X_val_combined)


In [ ]:
y_pred

array([0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0,
       0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0,
       0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0,
       1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0,
       0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0,
       0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0,
       0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1,
       1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0,
       0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0,
       0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0,

<h3 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
معیار ارزیابی
</font>
</h3>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    معیاری که برای ارزیابی عملکرد مدل انتخاب کرده‌ایم، <code>f1_score</code> نام دارد.
    <br>
    این معیار، سنجه ارزیابی کیفیت مدل شماست. به عبارت بهتر در سامانه داوری هم از همین معیار برای نمره‌دهی استفاده شده است.
    <br>
    پیشنهاد می‌شود با توجه به این معیار، عملکرد مدل خود را بر روی دادگان آموزش یا اعتبارسنجی ارزیابی کنید.
</font>
</p>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font color="red"><b color='red'>توجه:</b></font>
<font face="vazirmatn" size=3>
 جهت کسب امتیاز کامل نیاز است تا پاسخ شما حداقل امتیاز <code>80</code> را با توجه به معیار معرفی‌شده کسب نماید.
</font>
</p>

# Evaluation

In [ ]:
# Evaluate model
print("Binary F1-score (Real = positive class):",
      round(f1_score(y_val, y_pred), 4))

print("Weighted F1-score:",
      round(f1_score(y_val, y_pred, average='weighted'), 4))

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_pred,
    target_names=['Fake', 'Real']
))

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred))

Binary F1-score (Real = positive class): 1.0
Weighted F1-score: 1.0

Classification Report:
              precision    recall  f1-score   support

        Fake       1.00      1.00      1.00       247
        Real       1.00      1.00      1.00       128

    accuracy                           1.00       375
   macro avg       1.00      1.00      1.00       375
weighted avg       1.00      1.00      1.00       375

Confusion Matrix:
[[247   0]
 [  0 128]]


Checking for Potential Target Leakage

In [ ]:
# Check whether news type reveals the target label
print("Type values:")
print(train_data['type'].value_counts())

print("\nType vs. label:")
print(pd.crosstab(
    train_data['type'],
    train_data['label'],
    normalize='index'
).round(3))

print("\nUnique authors:", train_data['author'].nunique())
print("Unique domains:", train_data['site_domain'].nunique())

Type values:
type
bs            423
conspiracy    358
bias          315
hate          197
satire        112
junksci        82
fake           13
Name: count, dtype: int64

Type vs. label:
label       Fake  Real
type                  
bias         0.0   1.0
bs           1.0   0.0
conspiracy   1.0   0.0
fake         1.0   0.0
hate         0.0   1.0
junksci      1.0   0.0
satire       1.0   0.0

Unique authors: 418
Unique domains: 1


Evaluation Without the type Feature

In [ ]:
# Remove the target-leaking feature
categorical_features = ['author', 'site_domain']

# Rebuild preprocessing pipeline
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_features),
        ('num', numeric_transformer, numeric_features)
    ],
    sparse_threshold=1.0
)

# Fit preprocessing on training data only
X_train_tabular = preprocessor.fit_transform(X_train)
X_val_tabular = preprocessor.transform(X_val)

# Combine tabular and text features
X_train_combined = sp.hstack([
    sp.csr_matrix(X_train_tabular),
    sp.csr_matrix(text_train_reduced)
]).tocsr()

X_val_combined = sp.hstack([
    sp.csr_matrix(X_val_tabular),
    sp.csr_matrix(text_val_reduced)
]).tocsr()

# Train and evaluate SVM
svm_model = SVC(
    kernel='linear',
    C=1.0,
    class_weight='balanced'
)

svm_model.fit(X_train_combined, y_train)
y_pred = svm_model.predict(X_val_combined)

print("Binary F1-score:", round(f1_score(y_val, y_pred), 4))
print("Weighted F1-score:",
      round(f1_score(y_val, y_pred, average='weighted'), 4))

print("\n", classification_report(
    y_val, y_pred, target_names=['Fake', 'Real']
))
print("Confusion matrix:\n", confusion_matrix(y_val, y_pred))

Binary F1-score: 0.8487
Weighted F1-score: 0.9021

               precision    recall  f1-score   support

        Fake       0.90      0.96      0.93       247
        Real       0.92      0.79      0.85       128

    accuracy                           0.90       375
   macro avg       0.91      0.88      0.89       375
weighted avg       0.90      0.90      0.90       375

Confusion matrix:
 [[238   9]
 [ 27 101]]


Analyzing the Relationship Between Author and Label

In [ ]:
# Check whether authors appear with only one target label
author_label_counts = train_data.groupby('author')['label'].nunique()

print("Authors associated with only one label:",
      (author_label_counts == 1).sum())

print("Total unique authors:", train_data['author'].nunique())

print("\nAuthors appearing with both labels:")
print((author_label_counts > 1).sum())

Authors associated with only one label: 409
Total unique authors: 418

Authors appearing with both labels:
9


Evaluation Without the type & author Feature

In [ ]:
# Use only temporal and text features
categorical_features = []

numeric_features = ['year', 'month', 'hour', 'day_of_week', 'weekend']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features)
    ],
    remainder='drop'
)

# Fit on training data only
X_train_tabular = preprocessor.fit_transform(X_train)
X_val_tabular = preprocessor.transform(X_val)

# Combine temporal and text features
X_train_combined = sp.hstack([
    sp.csr_matrix(X_train_tabular),
    sp.csr_matrix(text_train_reduced)
]).tocsr()

X_val_combined = sp.hstack([
    sp.csr_matrix(X_val_tabular),
    sp.csr_matrix(text_val_reduced)
]).tocsr()

# Train SVM
svm_model = SVC(
    kernel='linear',
    C=1.0,
    class_weight='balanced'
)

svm_model.fit(X_train_combined, y_train)
y_pred = svm_model.predict(X_val_combined)

print("Binary F1-score:", round(f1_score(y_val, y_pred), 4))
print("Weighted F1-score:",
      round(f1_score(y_val, y_pred, average='weighted'), 4))

print("\n", classification_report(
    y_val, y_pred, target_names=['Fake', 'Real']
))
print("Confusion matrix:\n", confusion_matrix(y_val, y_pred))

Binary F1-score: 0.6768
Weighted F1-score: 0.7747

               precision    recall  f1-score   support

        Fake       0.84      0.81      0.83       247
        Real       0.66      0.70      0.68       128

    accuracy                           0.77       375
   macro avg       0.75      0.75      0.75       375
weighted avg       0.78      0.77      0.77       375

Confusion matrix:
 [[201  46]
 [ 39  89]]


Testing SVM Without Text Dimensionality Reduction

In [ ]:
# Combine temporal and original text features
X_train_full_text = sp.hstack([
    sp.csr_matrix(X_train_tabular),
    text_train
]).tocsr()

X_val_full_text = sp.hstack([
    sp.csr_matrix(X_val_tabular),
    text_val
]).tocsr()

print("Training shape:", X_train_full_text.shape)
print("Validation shape:", X_val_full_text.shape)

# Train linear SVM without dimensionality reduction
svm_full_text = SVC(
    kernel='linear',
    C=1.0,
    class_weight='balanced'
)

svm_full_text.fit(X_train_full_text, y_train)
y_pred_full_text = svm_full_text.predict(X_val_full_text)

print("\nBinary F1-score:",
      round(f1_score(y_val, y_pred_full_text), 4))

print("Weighted F1-score:",
      round(f1_score(y_val, y_pred_full_text, average='weighted'), 4))

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_pred_full_text,
    target_names=['Fake', 'Real']
))

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_full_text))

Training shape: (1125, 42146)
Validation shape: (375, 42146)

Binary F1-score: 0.6638
Weighted F1-score: 0.7879

Classification Report:
              precision    recall  f1-score   support

        Fake       0.81      0.90      0.85       247
        Real       0.75      0.59      0.66       128

    accuracy                           0.79       375
   macro avg       0.78      0.75      0.76       375
weighted avg       0.79      0.79      0.79       375

Confusion Matrix:
[[222  25]
 [ 52  76]]



Comparing Class Weights

In [ ]:
# Train SVM without class weighting
svm_unbalanced = SVC(
    kernel='linear',
    C=1.0,
    class_weight=None
)

svm_unbalanced.fit(X_train_full_text, y_train)
y_pred_unbalanced = svm_unbalanced.predict(X_val_full_text)

print("Binary F1-score:",
      round(f1_score(y_val, y_pred_unbalanced), 4))

print("Weighted F1-score:",
      round(f1_score(y_val, y_pred_unbalanced, average='weighted'), 4))

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_pred_unbalanced,
    target_names=['Fake', 'Real']
))

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_unbalanced))

Binary F1-score: 0.6484
Weighted F1-score: 0.7845

Classification Report:
              precision    recall  f1-score   support

        Fake       0.80      0.92      0.85       247
        Real       0.78      0.55      0.65       128

    accuracy                           0.79       375
   macro avg       0.79      0.74      0.75       375
weighted avg       0.79      0.79      0.78       375

Confusion Matrix:
[[227  20]
 [ 57  71]]


Tuning the C Parameter

In [ ]:
# Compare different C values
results = []

for c_value in [0.1, 1.0, 10.0]:
    model = SVC(
        kernel='linear',
        C=c_value,
        class_weight='balanced'
    )

    model.fit(X_train_full_text, y_train)
    predictions = model.predict(X_val_full_text)

    results.append({
        'C': c_value,
        'Binary F1 (Real)': f1_score(y_val, predictions),
        'Weighted F1': f1_score(
            y_val, predictions, average='weighted'
        )
    })

results_df = pd.DataFrame(results)
print(results_df.round(4))

      C  Binary F1 (Real)  Weighted F1
0   0.1            0.5329       0.6309
1   1.0            0.6638       0.7879
2  10.0            0.6545       0.7876


RBF: Comparing Linear and RBF Kernels

In [ ]:
# Compare linear and RBF kernels
kernel_results = []

for kernel_name in ['linear', 'rbf']:
    model = SVC(
        kernel=kernel_name,
        C=1.0,
        class_weight='balanced'
    )

    model.fit(X_train_full_text, y_train)
    predictions = model.predict(X_val_full_text)

    kernel_results.append({
        'Kernel': kernel_name,
        'Binary F1 (Real)': f1_score(y_val, predictions),
        'Weighted F1': f1_score(
            y_val, predictions, average='weighted'
        )
    })

kernel_results_df = pd.DataFrame(kernel_results)
print(kernel_results_df.round(4))

   Kernel  Binary F1 (Real)  Weighted F1
0  linear            0.6638       0.7879
1     rbf            0.6113       0.6957


Evaluating Temporal Features

In [ ]:
# Train SVM using text features only
svm_text_only = SVC(
    kernel='linear',
    C=1.0,
    class_weight='balanced'
)

svm_text_only.fit(text_train, y_train)
y_pred_text_only = svm_text_only.predict(text_val)

print("Text-only Binary F1:",
      round(f1_score(y_val, y_pred_text_only), 4))

print("Text-only Weighted F1:",
      round(f1_score(y_val, y_pred_text_only, average='weighted'), 4))

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_pred_text_only,
    target_names=['Fake', 'Real']
))

Text-only Binary F1: 0.652
Text-only Weighted F1: 0.7817

Classification Report:
              precision    recall  f1-score   support

        Fake       0.80      0.90      0.85       247
        Real       0.75      0.58      0.65       128

    accuracy                           0.79       375
   macro avg       0.78      0.74      0.75       375
weighted avg       0.78      0.79      0.78       375



Final Model Selection and Training on the Full Dataset

In [ ]:
# Train the final model on all training text vectors
final_model = SVC(
    kernel='linear',
    C=1.0,
    class_weight='balanced'
)

final_model.fit(train_text_vectors, y)

print("Final model trained successfully.")
print("Training samples:", train_text_vectors.shape[0])
print("Text features:", train_text_vectors.shape[1])

Final model trained successfully.
Training samples: 1500
Text features: 42141


Final Evaluation

In [ ]:
# Train the selected model on the training split
evaluation_model = SVC(
    kernel='linear',
    C=1.0,
    class_weight='balanced'
)

evaluation_model.fit(text_train, y_train)

# Predict validation labels
y_val_pred = evaluation_model.predict(text_val)

# Final evaluation metrics
print("Final Validation Results")
print("=" * 40)

print("Accuracy:",
      round(evaluation_model.score(text_val, y_val), 4))

print("Binary F1-score (Real):",
      round(f1_score(y_val, y_val_pred), 4))

print("Weighted F1-score:",
      round(f1_score(y_val, y_val_pred, average='weighted'), 4))

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_val_pred,
    target_names=['Fake', 'Real']
))

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))

Final Validation Results
Accuracy: 0.7893
Binary F1-score (Real): 0.652
Weighted F1-score: 0.7817

Classification Report:
              precision    recall  f1-score   support

        Fake       0.80      0.90      0.85       247
        Real       0.75      0.58      0.65       128

    accuracy                           0.79       375
   macro avg       0.78      0.74      0.75       375
weighted avg       0.78      0.79      0.78       375

Confusion Matrix:
[[222  25]
 [ 54  74]]


<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
 پیش‌بینی برای داده تست و خروجی
</font>
</h2>

<p dir=rtl style="direction: rtl;text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazirmatn" size=3>
    پس از مهندسی ویژگی و مدلسازی، الگوریتمی دارید که می‌تواند شما را از متغیرهای مستقل به متغیر هدف برساند.
    <br>
    از این مدل برای پیش‌بینی نمونه‌های موجود در داده تست استفاده کنید و نتایج را در قالب جدول (<code>dataframe</code>) زیر آماده کنید.
</font>
</p>

<div dir=rtl style="direction: rtl;text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazirmatn" size=3>
    
|ستون|توضیحات|
|------|---|
|label|برچسب پیشبینی شده|
    
</font>
</div>



<p dir=rtl style="direction: rtl;text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazirmatn" size=3>
    اسم دیتافریم باید <i>submission</i> باشد؛ در غیر این صورت، سامانه داوری نمی‌تواند تلاش‌ شما را ارزیابی کند.
    <br>
    این دیتافریم تنها شامل ۱ ستون با اسم <i>label</i> است و ۳۴۶ سطر دارد.
    <br>
    به ازای هر سطر موجود در دیتافریم <i>test</i> شما باید یک مقدار پیشبینی شده داشته باشید.
    <br>
    جدول زیر، ۵ سطر ابتدایی دیتافریم <code>submission</code> را نشان می‌دهد. البته در جواب شما، اعداد ستون <i>label</i> ممکن است متفاوت باشد.
</font>
</p>

<div style="text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazirmatn" size=3>
    
||label|
|----|-----|
|0|Fake|
|1|Fake|
|2|Real|
|3|Fake|
|4|Real|
</font>
</div>



In [ ]:
# Predict labels for the test set
test_predictions = final_model.predict(test_text_vectors)

# Convert numeric predictions to original labels
predicted_labels = pd.Series(test_predictions).map({
    0: 'Fake',
    1: 'Real'
})

# Create submission file
submission = pd.DataFrame({
    'label': predicted_labels
})

submission.to_csv(
    data_path + 'submission.csv',
    index=False
)

print("Submission file saved successfully.")
print("\nPrediction counts:")
print(submission['label'].value_counts())

print("\nSubmission preview:")
print(submission.head())
print("\nSubmission shape:", submission.shape)

Submission file saved successfully.

Prediction counts:
label
Fake    237
Real    109
Name: count, dtype: int64

Submission preview:
  label
0  Fake
1  Fake
2  Real
3  Real
4  Fake

Submission shape: (346, 1)


<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
<b>سلول جواب‌ساز</b>
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    برای ساخته‌شدن فایل <code>result.zip</code> سلول زیر را اجرا کنید. توجه داشته باشید که پیش از اجرای سلول زیر تغییرات اعمال شده در نت‌بوک را ذخیره کرده باشید (<code>ctrl+s</code>) تا در صورت نیاز به پشتیبانی امکان بررسی کد شما وجود داشته باشد.
</font>
</p>

In [ ]:
import zipfile
import joblib

def compress(file_names):
    print("File Paths:")
    print(file_names)
    compression = zipfile.ZIP_DEFLATED
    with zipfile.ZipFile("result.zip", mode="w") as zf:
        for file_name in file_names:
            zf.write('./' + file_name, file_name, compress_type=compression)

joblib.dump(model, 'model')
submission.to_csv('submission.csv', index=False)
file_names = ['fake_news_detection.ipynb', 'submission.csv', 'model']
compress(file_names)

File Paths:
['fake_news_detection.ipynb', 'submission.csv', 'model']


FileNotFoundError: [Errno 2] No such file or directory: './fake_news_detection.ipynb'